# Calculating *Optical Spectra* with **`exciting`** using *BSE*
By: Elias Richter

**<span style="color:firebrick">Note</span>**: Due to time constraints, all results used in this tutorial have been precomputed. Therefore, the corresponding data are retrieved from the [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries) database. Otherwise, `exciting` would need to be installed and compiled, and the appropriate environment would need to be configured. More information about exciting and its usage can be found on [exciting webpage](https://www.exciting-code.org/).

<hr style="border:2px solid #DDD"> </hr>

In [ ]:
from ase import Atoms
from excitingtools.input import ExcitingInputXML, ExcitingXSInput
from pathlib import Path
from IPython.display import Image


### <span style="color:#15317E">1. Theoretical Background: Bethe-Salpeter Equation</span>

Before starting with the calculation, we briefly review the theoretical background. Many-body perturbation theory (**MBPT**) offers powerful tools such as the Bethe-Salpeter equation (**BSE**) formalism to compute bound two-particle states. In the context of optical absorption, the **BSE** formalism is used to describe electron-hole pair excitations. Of particular interest are bound electron-hole pairs with energies below the band gap, which are quasi-particles referred to as **<span style="color:firebrick">excitons</span>**.

**The BSE Hamiltonian**

The **BSE** can be stated as an eigenvalue problem of an effective two-particle Hamiltonian:
    
$$
H ^{\rm (eff)}\: | A_{\lambda} \rangle= E_{\lambda}\: |A_{\lambda}\rangle\:,
$$
    
where the effective Hamiltonian is defined as
    
$$
H ^{\rm (eff)} = H ^{\rm (diag)}+2\:\gamma_{\rm x}\:H ^{\rm (x)} + \gamma_{\rm c}\:H ^{\rm (c)}\:.
$$
    
In this expression, three terms can be recognized, each carrying a specific physical meaning which will be clarified in the following. Moreover, the factors $\gamma_{\rm x}$ and $\gamma_{\rm c}$ allow one to choose different levels of approximation and to distinguish between **spin-singlet** (<em>i.e.</em>, $\gamma_{\rm x} \equiv 1$ and $\gamma_{\rm c} \equiv 1$) and **spin-triplet** channels (<em>i.e.</em>, $\gamma_{\rm x} \equiv 0$ and $\gamma_{\rm c} \equiv 1$).

Since the **BSE** describes interacting electron-hole pairs, it is natural to introduce a two-particle basis which reflects the excitation (resonant terms, labeled **r**) and the de-excitation (anti-resonant terms, labeled **a**) of independent electron-hole pairs:
    
$$
\Upsilon^\text{r}_{\alpha,\mathbf{q}}(\mathbf{r},\mathbf{r}') = \psi_{v_\alpha \mathbf{k}^+_\alpha}\!(\mathbf{r})\, \psi^*_{c_\alpha \mathbf{k}^-_\alpha}\!(\mathbf{r}')\;\phantom{!} \\[4 mm]
\Upsilon^\text{a}_{\alpha,\mathbf{q}}(\mathbf{r},\mathbf{r}') = \psi_{c_\alpha \mathbf{k}^+_\alpha}\!(\mathbf{r})\, \psi^*_{v_\alpha \mathbf{k}^-_\alpha}\!(\mathbf{r}')\;.
$$
    
Here, $ \psi_{n\mathbf{k}}\!(\mathbf{r}) $ are quasi-particle states. The combined index $\alpha$ incorporates $v$, indicating valence states, $c$ indicating conduction states and $\mathbf{k}$ standing for the **k**-points. The (crystal)-momentum difference between electron and hole is denoted by $\mathbf{q}$, such that $\mathbf{k}^\pm = \mathbf{k} \pm \mathbf{q}/2$. In the case of optical excitations, no momentum transfer from photon to electron is considered ($\mathbf{q} = 0$). In the basis presented above, the Hamiltonian has a $2 \times 2$ block structure
    
$$
H^{\text{(eff)}} = 
\begin{pmatrix}
 H^\text{rr} & H^\text{ra} \\
 H^\text{ar} & H^\text{aa}
\end{pmatrix},
$$
    
with the eigenvectors split into a resonant and an anti-resonant part:
    
$$
H^{\text{(eff)}}
\begin{pmatrix}
 X_\lambda \\
 Y_\lambda
\end{pmatrix}
=
E_\lambda
\begin{pmatrix}
 X_\lambda \\
 Y_\lambda
\end{pmatrix}.
$$
    
We can formally write the eigenvector coefficients in the Dirac notation as
    
$$
\begin{align}
(X_\lambda)_\alpha &= \langle (v_\alpha \mathbf{k}^+_\alpha) (c_\alpha \mathbf{k}^-_\alpha)^* | A_\lambda \rangle \\[4 mm]
(Y_\lambda)_\alpha &= \langle (c_\alpha \mathbf{k}^+_\alpha) (v_\alpha \mathbf{k}^-_\alpha)^* | A_\lambda \rangle.
\end{align}
$$
    
In the **BSE** formalism it is common to adopt the **Tamm-Dancoff-approximation** (**TDA**) which neglects the coupling between the resonant and anti-resonant components of the Hamiltonian. For $\mathbf{q} = 0$, this approximation entails that only the $H^\text{rr}$ part has to be considered. In this way, the necessary computational effort is significantly reduced. In the following, we focus only on the resonant-resonant block of the **BSE** and discuss its terms.

The **diagonal term** of the Hamiltonian, $ H^{\rm (diag)} $, accounts for the contribution of independent particle transitions energies
    
$$
H^{\text{(diag)}}_{\alpha\,\beta}\!(\mathbf{q}) = (\epsilon_{c_\alpha\,\mathbf{k}^-_\alpha} - \epsilon_{v_\alpha\,\mathbf{k}^+_\alpha}) \,\delta_{\alpha\,\beta},
$$
    
where $\epsilon_{n\,\mathbf{k}} $ is the single-particle energy of the $n$-th band. 

The **exchange term**, is defined as
    
$$
H^{\text{(x)}}_{\alpha\,\beta}(\mathbf{q}) = \int \!\!d^{3}\mathbf{r}\!\!\int\!\! d^{3}\mathbf{r'}
\,\Upsilon^{\text{r}*}_{\alpha,\mathbf{q}}(\mathbf{r},\mathbf{r})
\,v(\mathbf{r,r'})
\,\Upsilon^\text{r}_{\beta,\mathbf{q}}(\mathbf{r}',\mathbf{r}')
$$
    
and describes the repulsive exchange interaction between the electron-hole pairs $\alpha$ and $\beta$. The pair $\beta$ annihilates at $\mathbf{r}'$ and the pair $\alpha$ is created at $\mathbf{r}$. The interaction is mediated by the bare Coulomb potential  $v(\mathbf{r,r'})$. For $\mathbf{q}=0$, only the short-range part of this term is included. 

Finally, the **direct term** $H^{\text{(c)}}$ reads
    
$$
H^{\text{(c)}}_{\alpha\,\beta}(\mathbf{q}) = -\int\!\! d^{3}\mathbf{r}\!\!\int\!\! d^{3}\mathbf{r'}
\,\Upsilon^{\text{r}*}_{\alpha,\mathbf{q}}(\mathbf{r},\mathbf{r}')
\,W(\mathbf{r,r'})
\,\Upsilon^\text{r}_{\beta,\mathbf{q}}(\mathbf{r},\mathbf{r}')\;.
$$
    
It describes the scattering of the electron-hole pair $\beta$ into the pair $\alpha$ due to the screened Coulomb potential $ W(\mathbf{r,r'}) $. It accounts for the attractive electron-hole interaction including the screened potential of the form:
    
$$
W(\mathbf{r}, \mathbf{r}') =  \int\!\! d^{3}\mathbf{r}'' \,v(\mathbf{r}, \mathbf{r}'')\, \varepsilon^{-1}\!(\mathbf{r}'',\mathbf{r}') \;,
$$
    
where the screening of the Coulomb potential $ {v} $ is given by the microscopic dielectric function $ \varepsilon^{-1} $, which is calculated here within the random-phase approximation (**RPA**).

In the limit of vanishing $\mathbf{q}$ (optical limit), one can obtain the **macroscopic transverse dielectric tensor**  $ \varepsilon^{ij}_{\text{M}}(\omega)$ from the **BSE** solutions. The imaginary part of this tensor yields the optical absorption spectrum, as expressed by
    
$$
\operatorname{Im} \left[\varepsilon^{ij}_{\text{M}}(\omega)\right] = \dfrac{4\pi^2}{V} \sum_{\lambda}\: t^*_{\lambda,i} t_{\lambda,j} \,\frac{1}{\pi \delta} \left( \frac{\delta^2}{(\omega-E_\lambda)^2+\delta^2} - \frac{\delta^2}{(\omega+E_\lambda)^2+\delta^2} \right)\;. 
$$

Thus, considering a diagonal element, the spectrum is constructed from scaled positive (negative) Lorentizian function of width $\delta$ centered around the positive (negative) **BSE** eingenenergies. The scaling factors are determined by the transition coefficients $ t_{\lambda, i} $ which are a sum of dipole-transition matrix elements weighted by the **BSE** eigenvectors:
    
$$
t^*_{\lambda, i}=\mathrm{i} \sum_{\alpha} \left( X^{\lambda}_{\alpha} + Y^{\lambda}_{\alpha} \right) \langle c_\alpha\,\mathbf{k}_\alpha|-\hat{r}_i|v_\alpha\,\mathbf{k}_\alpha\rangle\,,
$$
    
where, within the **TDA**, $ Y^\lambda $ = 0.

In the actual computation, the dipole matrix elements are replaced by momentum matrix elements obtained by using the corresponding canonical commutator relation.
</details>

### <span style="color:#15317E">3. Preliminary GS calculations</span>

As a preliminary step to calculate excited-state properties from BSE, a ground-state calculation has to be performed. In this tutorial we consider as an example **$\beta$-Ga<sub>2</sub>O<sub>3</sub>**.

An `exciting` input file needs to contain the <code><span style="color:green">structure</span></code> element in which we include the lattice parameter, basis vectors, and atomic positions and the <code><span style="color:green">groundstate</span></code> element where we include parameters responsible for **DFT** ground-state calculations. We also need so called species files in which we define the augmented plane-wave and local-orbital basis. Have in mind that the basis within species files needs to be converged.

We can read the ground-state input from file using the exciting python interface **excitingtools**. Details on this can be found in the [**excitingtools tutorial**](https://www.exciting-code.org/uploads/exciting/tutorial_notebooks/tutorial_excitingtools.html).

In [ ]:
exciting_input = ExcitingInputXML.from_xml('data/gs_input.xml')

The main output of the **DFT** calculation for subsequent excited states calculations are stored in **STATE.OUT** and **EFERMI.OUT**. 

Since we use the mock runner and the binary file **STATE.OUT** is not transferable between maschines and compilers, we dont fetch **DFT** results seperately and directly proceed with the **BSE** calculations. But we assume, we have the neccesary ground state calculation done, therefor we can skip the recomputation of the ground state by seting the following parameter in the input file. 

In [ ]:
exciting_input.groundstate.do = "skip"

### <span style="color:#15317E">4. BSE calculations</span>

We create an input file to perfom an excited-state **BSE** calculation. 

To perform an excited-state calculation we must include the  <code><span style="color:green">xs</span></code> element in the input file:

In [ ]:
bse = {'bsetype': 'singlet', 
       'nstlbse': [51, 60, 1, 10], 
       'distribute': False, 
       'solver': 'direct'}

xs = ExcitingXSInput(xstype = 'BSE', 
                     ngridk = [10, 10, 10], 
                     vkloff = [0.12, 0.23, 0.11], 
                     ngridq = [10, 10, 10], 
                     rgkmax = 7, 
                     nempty = 30, 
                     gqmax = 1.1, 
                     lmaxapwwf = 8, 
                     broad = 0.0037, 
                     scissor = 0.0957, 
                     tevout = True, 
                     tappinfo = True, 
                     energywindow = {'intv': [0.0, 1.0], 'points': 1000}, 
                     screening = {'screentype': 'full'},
                     BSE = bse,
                     qpointset=[[0.0, 0.0, 0.0]])

We add it to the excisting input file.

In [ ]:
exciting_input.xs = xs

Now, we generate a directory for the calculation to be run in

In [ ]:
bse_dir = Path("bse_tutorial")
bse_dir.mkdir(exist_ok=True)

and save the generated input for exciting in an `input.xml` file. 

In [ ]:
input_xml = bse_dir / "input.xml"
exciting_input.write(input_xml)

The generated input file looks like this

In [ ]:
print(input_xml.read_text())

To understand the attributes that appear in this section of the input file, we will go through all of them in order to clarify their meaning.
<details> <summary><span style="color:firebrick"><strong>$\Rightarrow$ click here</strong></span></summary>

* <code><span style="color:mediumblue">xstype</span></code> defines the type of excited-state calculation: here we set it to **BSE**;
* <code><span style="color:mediumblue">ngridk</span></code> within the <code><span style="color:green">xs</span></code> element is used in the non-self-consistent-field calculation required to obtain the single-particle eigenstates and eigenenergies. If not defined, it defaults to the **k**-point mesh used in the <code><span style="color:green">groundstate</span></code> calculation.  Calculations should be converged with respect to the **k**-mesh;
* <code><span style="color:mediumblue">vkloff</span></code> is used to shift the **k**-mesh <code><span style="color:mediumblue">off symmetry</span></code> by a small displacement in order to break all symmetry relations among the **k**-points of the mesh. In this way,  all the **k**-points in the mesh will be crystallographically inequivalent and there will be no redundant contribution to the spectrum;
* <code><span style="color:mediumblue">ngridq</span></code> defines the **q**-mesh for the calculation of the screening. It is a good practice to choose a **q**-mesh equivalent to the **k**-mesh;
* <code><span style="color:mediumblue">nempty</span></code> determines the number of empty states used in the construction of the BSE Hamiltonian;
* <code><span style="color:mediumblue">gqmax</span></code> is an energy threshold for the local field effects included in the calculation. The number of  **G**-vectors employed in the calculation is written in the first line of the file **GQPOINTS_QM001.OUT**. Tuning <code><span style="color:mediumblue">gqmax</span></code> may require adjusting the value of <code><span style="color:mediumblue">gmaxvr</span></code> in the <code><span style="color:green">groundstate</span></code> element and re-running the ground-state calculation;
* <code><span style="color:mediumblue">broad</span></code> defines the Lorentzian broadening $\delta$ for the calculated spectra;
* When the attribute <code><span style="color:mediumblue">scissor</span></code> is set different from zero, a scissors operator is applied to correct the band gap obtained from the ground state calculation in order to mimic the quasi-particle gap. This parameter should not appear when the electronic structure resulting from a **GW** calculation is taken as starting point for the **BSE** calculation;
* <code><span style="color:mediumblue">tappinfo</span></code> causes output to be appended to the log file **INFOXS.OUT** without overwriting the existing file.
* <code><span style="color:mediumblue">dtevouto</span></code> sets the energy output to electronvolt (eV).

Additional sub-elements appear inside the <code><span style="color:green">xs</span></code> element (refer to the **[Input Reference](https://www.exciting-code.org/home/about/input-reference)** for further details):
* <code><span style="color:green">energywindow</span></code> defines the energy window and the number of points for the calculation of the optical spectrum. The attribute <code><span style="color:mediumblue">intv</span></code> indicates the lower and upper energy limits for the calculation of the spectrum, while <code><span style="color:mediumblue">points</span></code> defines the number of energy points to be used. 
* <code><span style="color:green">screening</span></code> defines the parameters used to compute the screened Coulomb potential. The attribute <code><span style="color:mediumblue">screeningtype</span></code> selects the approximation for the screening: in this case we include the full screening in the calculation. For other options refer to the **[Input Reference](https://www.exciting-code.org/home/about/input-reference)**. The attribute <code><span style="color:mediumblue">nempty</span></code> defines the number of empty states to be included in the calculation of the screening matrix. Note that this is a different attribute than the one within the <code><span style="color:green">xs</span></code> element and must always be specified when performing a **BSE** calculation.
* In the <code><span style="color:green">BSE</span></code> element the actual parameters for a **BSE** calculation are set. The attribute <code><span style="color:mediumblue">bsetype</span></code> defines the level of approximation in the solution of the **BSE** Hamiltonian. Here we are calculating the **"singlet"** states, meaning that the full **BSE** Hamiltonian is diagonalized. Other options include the calculations of **"triplet"**, where the exchange term of the Hamiltonian is neglected, **RPA**, where the direct term is ignored, and the independent-particle (**IP**) approximation, where only the diagonal part of the Hamiltonian is considered. For further details see **[Input Reference](https://www.exciting-code.org/home/about/input-reference)**. The attribute <code><span style="color:mediumblue">nstlbse</span></code> defines the range of occupied and empty states included in the **BSE** calculation. The first two integers define the lowest and highest occupied bands, while the last two refer to the unoccupied bands. To define the band range we refer to the file **EIGVAL.OUT** obtained from the ground state calculation. Please notice that the occupied bands are numbered starting from the lowest occupied state, while unoccupied bands are numbered starting from the lowest unoccupied band.
* The element <code><span style="color:green">qpointset</span></code> is related to the dependence of the response function upon the momentum transfer along different directions (see tutorial **q-dependent TDDFT**). Here, the <code><span style="color:green">qpoint</span></code> is set to **"0.0 0.0 0.0"** since we are neglecting momentum transfer in this calculation ($\mathbf{q}=0$).


**<span style="color:firebrick">Note</span>**: Since DFT results are used as foundation, which typically underestimate the band-gap, we need to include a scissors operator specified by the attribute <code><span style="color:mediumblue">scissor</span></code>. The value is chosen in experimental agreement.

For further information on the **BSE** workflow within `exciting` <details> <summary><span style="color:firebrick"><strong>$\Rightarrow$ click here</strong></span></summary>

The work-flow for a **BSE** calculation is shown and described in the following:

<figure>
<img src="data/BSE_workflow.png" width="700" align="left" style="display:inline;margin:2px 2px 2px 10px;"/>
</figure><br>

1. As a first step, the <code><span style="color:green">xs</span></code> element triggers a **<span style="color:darkgreen">one-step groundstate calculation</span>** to generate a set of Kohn-Sham (**KS**) eigenvalues $\varepsilon_{n\mathbf{k}}$ and eigenfunctions $\psi_{n\mathbf{k}}$. This ground-state calculation uses a previously obtained **KS** potential (from a previous <code><span style="color:green">groundstate</span></code> run). It is important to note that the parameters for this one-step ground-state run are completely independent of those in the <code><span style="color:green">groundstate</span></code> element. In particular, the <code><span style="color:mediumblue">ngridk</span></code> and <code><span style="color:mediumblue">nempty</span></code> parameters that determine the range of $n\mathbf{k}$ indices are <code><span style="color:green">xs</span></code>\%<code><span style="color:mediumblue">ngridk</span></code> and <code><span style="color:green">xs</span></code>\%<code><span style="color:mediumblue">nempty</span></code>, respectively. These parameters are independent, and do not conflict with <code><span style="color:green">groundstate</span></code>\%<code><span style="color:mediumblue">ngridk</span></code> and <code><span style="color:green">groundstate</span></code>\%<code><span style="color:mediumblue">nempty</span></code>.

2. Next, the **<span style="color:darkgreen">momentum matrix elements</span>** $p_{i, n m \mathbf{k}}=\langle \psi_{n\mathbf{k}}|\hat p_{i}|\psi_{m\mathbf{k}} \rangle$ are calculated. The size of this matrix is determined by the number of **KS** orbitals obtained in the previous step. The matrix elements are used later in the work-flow for calculating the coefficients $t_{\lambda,i}$ defined in **<span style="color:darkred">Section 1</span>**.

3. Then, the static **<span style="color:darkgreen">dielectric matrix</span>** in the **RPA** approximation, $\varepsilon^\textrm{RPA}_{\mathbf{G,G'}}(\mathbf{q})$, is calculated. This calculation involves three important intermediate steps: (<em>i</em>) A new one-step ground state calculation is performed, using the same KS potential as in the first step. The parameters for this additional ground state calculation are independent of those in the <code><span style="color:green">xs</span></code> and <code><span style="color:green">groundstate</span></code> elements. There is however a requirement: <code><span style="color:green">screening</span></code>\%<code><span style="color:mediumblue">nempty</span></code> $\ge$ <code><span style="color:green">xs</span></code>\%<code><span style="color:mediumblue">nempty</span></code>. The reason for this is that the calculation of the screening must involve typically many more empty **KS** orbitals than those determining the electron-hole pairs in the definition of the **BSE** Hamiltonian, which usually involve a small number of empty bands. (<em>ii</em>) For this newly generated set of **KS** orbitals, the momentum matrix elements $p_{i, n m \mathbf{k}}$ are also calculated and written into a separate file. (<em>iii</em>) The independent particle density response function $\chi_0$ is assembled using the momentum matrix elements obtained in (<em>ii</em>) and plane-wave matrix elements calculated on-the-fly and not subsequently written into files. Finally, the dielectric matrix is computed as $\varepsilon^{RPA}=\mathbf{1}-v\chi_0$ and written into separate files for each $\mathbf{q}$.

4. Next the **<span style="color:darkgreen">direct term of the BSE Hamiltonian</span>** is calculated. This is done by first reading  $\varepsilon^{RPA}$ from the files generated in the previous step, then inverting it to build the Fourier transform of the screened Coulomb interaction matrix $W_{\mathbf{G,G'}}(\mathbf{q})$, and finally building the matrix $H^{\rm(c)}_{\alpha\,\beta}$. The **KS** orbitals used to build this matrix are those obtained in the first step, and the band ranges for valence ($v$) and conduction ($c$) band indices are determined by the attribute <code><span style="color:mediumblue">nstlbse</span></code>. This means that the number of empty states generated in the first step (<em>i.e.</em>, <code><span style="color:green">xs</span></code>\%<code><span style="color:mediumblue">nempty</span></code>) must be equal or larger than the highest index for conduction bands set in the attribute <code><span style="color:mediumblue">nstlbse</span></code>.

5. Next the **<span style="color:darkgreen">exchange term of the BSE Hamiltonian</span>**,  $H^{\rm (x)}_{\alpha\,\beta}$, is obtained. For its calculation only plane-wave matrix elements between **KS** orbitals calculated in **<span style="color:darkred">step 1</span>** are used. These matrix elements are calculated on the fly and not stored in any file. The valence and conduction band ranges are defined by the attribute <code><span style="color:mediumblue">nstlbse</span></code>, as in the previous step.

6. Finally, **<span style="color:darkgreen">BSE effective Hamiltonian</span>** $H^{\rm(eff)}_{\alpha\,\beta}$ is built and diagonalized. The band ranges are defined by the attribute  <code><span style="color:mediumblue">nstlbse</span></code>. Once the **BSE** effective Hamiltonian is diagonalized, the **<span style="color:darkgreen">macroscopic dielectric function</span>** (and other derived quantities) is calculated using the expression given in **<span style="color:darkred">Section 1</span>**. The momentum matrix elements used for the evaluation of the coefficients $t_{\lambda,i}$ are those calculated in **<span style="color:darkred">step 2</span>**.

The execution of these steps can be controlled directly from the input file by adding the <code><span style="color:green">plan</span></code> element to the <code><span style="color:green">xs</span></code> block and specifying which task should be run. The standard **BSE** execution follows implicitly the plan reported below:

```xml
...
   <xs...>
 
      <plan>
         <doonly task="xsgeneigvec"/>
         <doonly task="writepmatxs"/>
         <doonly task="scrgeneigvec"/>
         <doonly task="scrwritepmat"/>
         <doonly task="screen"/>
         <doonly task="scrcoulint"/>
         <doonly task="exccoulint"/>
         <doonly task="bse"/>
      </plan>

   </xs>                 
...

```
</details>

**<span style="color:firebrick">Note</span>**: Since the input file corresponds to higher converged results, the corresponding calculations are computational demanding. In reality they need to be paralized properly.

With that, we can run exciting using **excitingscripts**. This python library contains a lot of helpful scripts for running exciting and postprocessing exciting outputs. In this tutorial, we use the mock runner to fetch the pre-computed data from [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries).

In [ ]:
%%bash
cd bse_tutorial
python -m excitingscripts.dpg2026.mock_execute_single BSE --overwrite
cd ..

Now, that the calculation would have finished, a number of files and folders would be present in the working directory. Most of them contain technical information about the calculation that is not strictly related to the physical interpretation of the results. We are interested in plotting the optical absorption spectrum, given by the imaginary part of its macroscopic dielectric function. Therefore, in order to proceed with the analysis of the results, we considered only the data contained in the files starting with `EPSILON`. 

In order to plot the optical absoroption spectrum, execute the following commands from the excitingscripts library:

In [ ]:
%%bash
cd bse_tutorial
python -m excitingscripts.plot.files -f EPSILON_BSE-singlet-TDA-BAR_SCR-full_OC11.OUT EPSILON_BSE-singlet-TDA-BAR_SCR-full_OC22.OUT EPSILON_BSE-singlet-TDA-BAR_SCR-full_OC33.OUT -ll '$\epsilon_M^{xx}$'  '$\epsilon_M^{yy}$' '$\epsilon_M^{zz}$' -lx 'Energy [eV]'  -ly 'Im $\varepsilon_M$'  -t 'Macroscopic dielectric function'  -g  -rc  -cy 3  -x 0 27 
cd ..

We plot the absorption spectrum along the three axes:

In [ ]:
Image(filename='bse_tutorial/PLOT.png')

**<span style="color:firebrick">Note</span>**: The above command plots the imaginary part (indicated by the i after the filename) of the macroscopic dielectric function, contained in the third column, versus energy in eV, in the first column. The file `EPSILON_BSE-singlet-TDA-BAR_SCR-full_OC11.OUT` also stores the real part of the macroscopic dielectric function calculated directly and via a Kramers-Kronig transformation of the imaginary part in the second and fourth columns, respectively.

<hr style="border:2px solid #DDD"> </hr>
This concludes our tutorial for the computation of optical spectra with BSE. You are encouraged to checkout the other methods implemented in exciting via the tutorials in this suite.
<hr style="border:2px solid #DDD"> </hr>

### <span style="color:#15317E">5. Additional Information</span>

**BSE on top of GW**: 

Since density functional theory (DFT) provides Kohn-Sham eigenvalues describe fictitious non-interacting electrons and do not account for the true excitation energies it also typically underestimates band gaps, especially when semi-local xc-functionals (e.g. PBE) are used. Since the **BSE** builds explicitly on single-particle energies and wave functions, inaccuracies at the **DFT** level directly propagate into the excitation spectrum.

The **GW** approximation corrects the single-particle energies by incorporating a non-local, energy-dependent self-energy operator, yielding accurate quasiparticle band structures and band gaps. Using **GW** as foundation ensures that the electron–hole interaction treated within **BSE** leads to on a realistic quasiparticle spectrum, with significantly improved excitation energies, exciton binding energies, and optical spectra, establishing  a consistent many-body perturbation theory framework for describing excited-state properties. For details regarding **GW** and the calculation we refer to the dedicated tutorial and **[exciting-webpage tutorial](http://exciting.wikidot.com/neon-electronic-bandstructure-from-gw)**. 

In order to use the quasi-particle energies computed from **GW** as starting point for the BSE calculation, it is sufficient to include in the input file the <code><span style="color:green">gw</span></code> element. With this option the electronic eigenstates and eigenenergies are read from the **GW** output and the attribute <code><span style="color:mediumblue">scissor</span></code> is automatically ignored.

**Scaling and Convergence:** 

**BSE** calculations are extremely demanding, even on state-of-the-art computer infrastructure. The scaling with respect to the size of the **k**-mesh is **quadratic** with respect to the setup of the Hamlitonian (direct term). Moreover, for a full diagonalization to the eigenvalue problem the scalability goes like the **third power** of the **k**-mesh. An estimate for the scaling of the screened Coulomb interaction, which enters the direct term of the BSE Hamiltonian, and the full **BSE** eigenvalue problem is the 
following

$$
T_{\rm BSE} \sim \alpha_{\rm SCR}(N_v N_c N_{\bf k}) N_{\bf q}N_{\bf G}^2 + \alpha_{\rm HAM}(N_v N_c N_{\bf k})^2 N_{\bf G}^2 + \alpha_{\rm DIAG} (N_v N_c N_{\bf k})^3 
$$

where $N$ stands for the "number of" and the subscripts denote the **k**-points, **q**-points, **G**-vectors, valence and conduction states. Crucial convergence parameters for **BSE** within exciting are <code><span style="color:mediumblue">ngridk</span></code>, <code><span style="color:mediumblue">nempty</span></code>, <code><span style="color:mediumblue">gqmax</span></code>, and <code><span style="color:mediumblue">nstlbse</span></code>.

**FastBSE:** 

Newell extensions of exciting propose a low scaling alternative of the direct BSE solution. **fastBSE** is low scaling, matrix-free, iterative algorithm to solve the Bethe-Salpeter equation (**BSE**). It scales with $\mathcal O(N_v N_c(1  + \log N_k))$ yielding a massive speed up compared to the **direct** algorithm that scales with $\mathcal O(N_v^3 N_c^3 N_k^3)$.
Setting <code><span style="color:mediumblue">solver</span>=<span style="color:firebrick">"fastBSE"</span></code> within the <code><span style="color:green">xs</span></code> element tells exciting to use the **fastBSE** solver. Further details are proposed in the dedicated **[fastBSE tutorial](https://www.exciting-code.org/uploads/exciting/tutorial_notebooks/optical_spectra_from_fastBSE.html)**.


**BSE beyond the TDA:** 

To go beyond the Tamm-Dancoff approximation we need to add <code><span style="color:mediumblue">coupling</span>=<span style="color:firebrick">"true"</span></code> in the <code><span style="color:green">BSE</span></code> element of the input file. Compared to the previous calculations within the **TDA** as described above, now also the resonant-anti-resonant coupling block of the screened Coulomb interaction needs to be recalculated, since the Fourier coefficients of this block are needed at different **k**-points than those of the resonant-resonant block, increasing the computational costs. No additional action is required for the exchange interaction. 

### <span style="color:#15317E">Literature</span>

- **<span style="color:firebrick">[1]</span>** The **BSE** has been introduced for the first time in E.E. Salpeter and H.A. Bethe, Phys. Rev. **84**, 1232 (1951).
- **<span style="color:firebrick">[2]</span>** Details on implementation and application of the **BSE** formalism in **exciting**: Stephan Sagmeister and Claudia Ambrosch-Draxl, <em>Time-dependent density functional theory versus Bethe-Salpeter equation: An all-electron study</em>, Phys. Chem. Chem. Phys. **11**, 4451 (2009) [**PDF**](https://pubs.rsc.org/en/content/articlelanding/2009/CP/b903676h).
- **<span style="color:firebrick">[3]</span>** More details on the implementation of the **BSE formalism** within the LAPW method and applications to organic semiconductors: S. Sagmeister, PhD thesis, University of Graz, August 2009 [**PDF**](https://www.box.net/s/rpqegscqxr).
- Additional information on the **BSE** implementation beyond the **TDA** and zero momentum transfer can be found in B. Aurich's Master thesis [**PDF**](http://exciting.wdfiles.com/local--files/oxygen-excited-states-from-bse/BSEtheory.pdf).
- **<span style="color:firebrick">[4]</span>** <em>Application of the Green’s functions method to the study of the optical properties of semiconductors</em>, G. Strinati, Riv. Nuovo Cim. (1988) [**PDF**](https://link.springer.com/article/10.1007/BF02725962).
- **<span style="color:firebrick">[5]</span>** <em>Beyond the Tamm-Dancoff approximation for extended systems using exact diagonalization</em>, T. Sander, <em>et al</em> (2015) [**PDF**](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.92.045209).
- **<span style="color:firebrick">[6]</span>** <em>Electronic excitations: density-functional versus many-body Green's-function approaches</em>, G. Onida, <em>et al</em> (2015) [**PDF**](https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.74.601).
- **<span style="color:firebrick">[7]</span>** <em> Fast optical absorption spectra calculations for periodic solid state systems</em>, F. Henneke, <em>et al</em> (2020) [**PDF**](https://msp.org/camcos/2020/15-1/camcos-v15-n1-p04-s.pdf) 
- **<span style="color:firebrick">[8]</span>** <em> Low scaling BSE implementation in the exciting code</em>, B. Maurer, <em>et al</em> (2026) [**PDF**](https://www.theoj.org/joss-papers/joss.08866/10.21105.joss.08866.pdf) 

<hr style="border:2px solid #DDD"> </hr>